# PA 766 · Module 7 · Language models, tokens, and embeddings

## Setup

Run the install cell in a fresh Python 3 session. Model/tokenizer downloads require internet. Phi-3 generation requires a CUDA GPU here; in Colab, select **Runtime → Change runtime type → T4 GPU**. Token inspection and the smaller embedding models also work on CPU.

In [1]:
# Run once in a fresh environment; restart the kernel if packages were already imported.
%pip install -q transformers sentence-transformers accelerate pandas torch sentencepiece protobuf

In [2]:
import html as html_library
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

In [4]:
SEED = 42  # Choose a seed for repeatable results
random.seed(SEED)  # Set Python's random seed
np.random.seed(SEED)  # Set NumPy's random seed
torch.manual_seed(SEED)  # Set PyTorch's random seed

HAS_CUDA = torch.cuda.is_available()  # Check whether a CUDA GPU is available
RUNTIME_DEVICE = "cuda" if HAS_CUDA else "cpu"  # Choose GPU or CPU

# Keep True to run the tokenizer and embedding examples.
RUN_REMOTE_MODELS = True  # Enable examples that download models

# Phi-3 is large enough that we only load it automatically on a CUDA GPU.
RUN_LARGE_LLM = HAS_CUDA  # Run Phi-3 only when a CUDA GPU is available

print(f"PyTorch device: {RUNTIME_DEVICE}")  # Show the selected device
print(f"Run remote model examples: {RUN_REMOTE_MODELS}")  # Show whether remote models are enabled
print(f"Run Phi-3 generation examples: {RUN_LARGE_LLM}")  # Show whether Phi-3 is enabled

PyTorch device: cuda
Run remote model examples: True
Run Phi-3 generation examples: True


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Part I. From a prompt to generated text

A **tokenizer** converts text to token IDs. A **language model** predicts the next token repeatedly to generate text. Load compatible tokenizer/model checkpoints; the same ID can mean different things in different vocabularies.

[Phi-3 mini](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) is the original notebook's generation model. Only its model weights require a GPU in this notebook; its tokenizer loads on CPU as well.

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline  # Import model tools

LLM_CHECKPOINT = "microsoft/Phi-3-mini-4k-instruct"  # Name the model to use
llm = None  # Start without a loaded model
# Load the tokenizer even on CPU so token inspection remains available.
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_CHECKPOINT)  # Load the tokenizer

if RUN_LARGE_LLM:  # Load the model only when a CUDA GPU is available
    llm = AutoModelForCausalLM.from_pretrained(  # Download and load Phi-3
        LLM_CHECKPOINT,  # Use the selected model
        device_map="cuda",  # Place it on the GPU
        dtype=torch.float16,  # Use less GPU memory; compatible with a Colab T4
        trust_remote_code=False,  # Do not run custom model code
    )
    print(f"Loaded {LLM_CHECKPOINT} on {llm.device}.")  # Confirm where it loaded
else:  # Run this when no CUDA GPU is available
    print(
        "Phi-3 was not loaded because a CUDA GPU was not detected. "
        "Use a GPU runtime and rerun this cell to enable the generation demos."
    )  # Explain how to enable generation

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded microsoft/Phi-3-mini-4k-instruct on cuda:0.


### 1.1 A generation pipeline

`apply_chat_template` inserts the model's user/assistant markers. The pipeline handles tokenization, generation, and decoding. Here `do_sample=True` samples varied responses; `temperature` and `top_p` control that sampling.

In [8]:
if RUN_LARGE_LLM:
    generator = pipeline(
        "text-generation",
        model=llm,
        tokenizer=llm_tokenizer,
        return_full_text=False,  # Return the response without repeating the prompt.
    )

    joke_messages = [
        {"role": "user", "content": "Create a funny joke about chickens."}
    ]
    # Format the conversation before sending it to the model.
    joke_prompt = llm_tokenizer.apply_chat_template(
        joke_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    joke_outputs = generator(
        joke_prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=1.1,
        top_p=0.95,
        num_return_sequences=3,
    )

    for number, output in enumerate(joke_outputs, start=1):
        print(f"Joke {number}:")
        print(output["generated_text"])
        print()
else:
    print("Generation skipped; set RUN_LARGE_LLM=True in a CUDA runtime.")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'top_p', 'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup a

Joke 1:
Why did the chicken go to the seance? To talk to the other side and see if the farmer left the grain behind!

Joke 2:
Why did the chicken join a band? Because it had the drumsticks for it!

Joke 3:
Why did the chicken cross the playground? To get to the other slide!



### 1.2 Generation without the pipeline

The same steps explicitly: format the prompt, tokenize it, generate IDs, and decode only the new tokens.

In [9]:
if RUN_LARGE_LLM:  # Run this example when Phi-3 is loaded
    email_messages = [  # Create the chat message
        {
            "role": "user",  # Tell the model this is a user request
            "content": (
                "Write an email apologizing to Sarah for the tragic gardening "
                "mishap. Explain how it happened."
            ),  # Give the model its writing task
        }
    ]
    email_prompt = llm_tokenizer.apply_chat_template(  # Format the message for Phi-3
        email_messages,
        tokenize=False,  # Return text rather than token IDs
        add_generation_prompt=True,  # Mark where the model should start replying
    )
    email_inputs = llm_tokenizer(  # Convert the prompt to token IDs
        email_prompt,
        return_tensors="pt",  # Return PyTorch tensors
    ).to(llm.device)  # Move the inputs to the model's device

    # Inference does not need gradients used for model training.
    with torch.inference_mode():  # Generate without tracking training gradients
        email_output_ids = llm.generate(  # Generate a response
            **email_inputs,  # Pass in the tokenized prompt
            max_new_tokens=350,  # Generate at most 350 new tokens
            do_sample=True,  # Choose tokens using sampling
            temperature=1.1,  # Allow more varied wording
            top_p=0.95,  # Sample from the most likely tokens
            num_return_sequences=1,  # Generate one email
            pad_token_id=llm_tokenizer.eos_token_id,  # Set the padding token
        )

    prompt_length = email_inputs["input_ids"].shape[1]  # Count the prompt tokens

    for number, output_ids in enumerate(email_output_ids, start=1):  # Process each result
        # Remove the input IDs so the printed result contains only generated text.
        new_token_ids = output_ids[prompt_length:]  # Keep only new tokens
        generated_email = llm_tokenizer.decode(  # Convert tokens back to text
            new_token_ids,
            skip_special_tokens=True,  # Hide special formatting tokens
        )

        print(f"Email {number}:")  # Print the email number
        print(generated_email)  # Print the generated email
        print()  # Add a blank line
else:  # Run this when Phi-3 was not loaded
    print(
        "Direct generation skipped; "
        "set RUN_LARGE_LLM=True in a CUDA runtime."
    )  # Explain why generation was skipped

Email 1:
Subject: Heartfelt Apology for Our Garden Misadventure


Dear Sarah,


I hope this message finds you well.


I am writing to express my sincerest apologies for the unfortunate incident that occurred during our recent collaborative gardening effort on your exquisite front yard. I am deeply regretful to have contributed to the mishap, and the aftermath has weighed heavily on my conscience.


During our joint planting venture, we were both enveloped in the joy of our green thumb endeavors. It was there, amidst the flowers and the fresh soil, that I let my excitement override my usual prudence. In my zeal to secure the cherished roses you so dearly cherish to the right spot, I may have misjudged the soil's firmness. Alas, it was not as forgiving as I perceived, and the sudden digging motion led to an unforeseen disruption beneath the surface.


As I unknowingly toppled your old stone pathway, I recognized the gravity of my actions immediately. The path, a symbol of resilience for 

### Inspect token IDs

Token IDs are vocabulary lookup keys, not measurements of importance. The table shows each token's ID, vocabulary symbol, and decoded text fragment. This cell works without loading the generation model.

In [11]:
token_text = "The city should invest more in public transportation."  # Text to examine
prompt_ids = llm_tokenizer(token_text, add_special_tokens=False).input_ids  # Convert it to token IDs

In [ ]:
# Token boundaries can divide a word into smaller pieces.
token_table = pd.DataFrame(  # Build a table of the tokens
    {
        "position": range(len(prompt_ids)),  # Number each token's position
        "token_id": prompt_ids,  # Show each token's numeric ID
        "vocabulary_token": llm_tokenizer.convert_ids_to_tokens(prompt_ids),  # Show its vocabulary form
        "decoded_piece": [
            repr(llm_tokenizer.decode([token_id]))  # Show the text represented by one token
            for token_id in prompt_ids  # Repeat for every token
        ],
    }
)
display(token_table)  # Display the table

## Part II. Comparing tokenizers

Different vocabularies, casing rules, training data, and algorithms can split the same text differently. Each colored block below is one token; hover to see its ID. The example includes capitalization, Unicode, code, spaces, and arithmetic.

In [13]:
TOKEN_COLORS = [  # Colors used to highlight neighboring tokens
    "#66c2a5",
    "#fc8d62",
    "#8da0cb",
    "#e78ac3",
    "#a6d854",
    "#ffd92f",
]

def show_tokens(text, tokenizer_name):  # Display how a tokenizer splits text
    # Display tokenizer boundaries while making spaces and newlines visible.
    comparison_tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)  # Load the tokenizer
    token_ids = comparison_tokenizer(text, add_special_tokens=False).input_ids  # Get token IDs

    blocks = []  # Store the colored HTML blocks
    for position, token_id in enumerate(token_ids):  # Go through each token
        piece = comparison_tokenizer.decode([token_id])  # Convert it back to text
        visible_piece = (
            piece.replace(" ", "␠")  # Show spaces
            .replace("\t", "⇥")  # Show tabs
            .replace("\n", "↵\n")  # Show newlines
        )
        escaped_piece = html_library.escape(visible_piece) or "∅"  # Make it safe to display as HTML
        color = TOKEN_COLORS[position % len(TOKEN_COLORS)]  # Pick a color
        blocks.append(  # Add a colored block for this token
            f'<span title="token id: {token_id}" '
            f'style="background:{color}; color:#111; padding:3px 5px; '
            f'margin:2px; border-radius:4px; display:inline-block;">'
            f'{escaped_piece}</span>'
        )

    display(HTML("".join(blocks)))  # Display all the blocks together
    print(f"{tokenizer_name}: {len(token_ids)} tokens")  # Print the token count
    return token_ids  # Return the IDs for later use

In [ ]:
tokenizer_test_text = '''  # Create text with varied characters and spacing
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
'''.strip()  # Remove the leading and trailing newline

In [14]:
tokenizer_checkpoints = [
    "bert-base-uncased",
    "bert-base-cased",
    "gpt2",
    "google/flan-t5-small",
    "microsoft/Phi-3-mini-4k-instruct",
]

tokenizer_results = {}
if RUN_REMOTE_MODELS:
    for checkpoint in tokenizer_checkpoints:
        print(f"\n{checkpoint}")
        try:
            tokenizer_results[checkpoint] = show_tokens(
                tokenizer_test_text, checkpoint
            )
        except Exception as error:
            # A failed or restricted download should not stop the remaining comparisons.
            print(f"Unavailable: {type(error).__name__}: {error}")
else:
    print("Tokenizer downloads skipped because RUN_REMOTE_MODELS=False.")


bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

google/flan-t5-small: 56 tokens

microsoft/Phi-3-mini-4k-instruct


microsoft/Phi-3-mini-4k-instruct: 61 tokens


### Your Turn: Replace the test text above. See what you get

## Part III. Contextual token embeddings

A transformer produces a vector for each token position. These vectors depend on context: “bank” can refer to money or a river. Use the same checkpoint for the tokenizer and encoder so their vocabularies match.

In [15]:
from transformers import AutoModel

ENCODER_CHECKPOINT = "microsoft/deberta-v3-xsmall"
encoder_tokenizer = None
encoder_model = None

if RUN_REMOTE_MODELS:
    # Load the tokenizer and model from the same checkpoint.
    encoder_tokenizer = AutoTokenizer.from_pretrained(ENCODER_CHECKPOINT)
    encoder_model = AutoModel.from_pretrained(ENCODER_CHECKPOINT).to(RUNTIME_DEVICE)
    encoder_model.eval()  # Disable training-time behavior such as dropout.
    print(f"Loaded {ENCODER_CHECKPOINT} on {RUNTIME_DEVICE}.")
else:
    print("Contextual encoder download skipped.")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  241MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

Loaded microsoft/deberta-v3-xsmall on cuda.


In [18]:
embedding_table = None  # Start without an embedding table

if encoder_model is not None:  # Continue only if the encoder is loaded
    encoder_inputs = encoder_tokenizer("Hello world", return_tensors="pt")  # Convert text to tensors
    encoder_inputs = {
        name: tensor.to(RUNTIME_DEVICE)  # Move each tensor to the selected device
        for name, tensor in encoder_inputs.items()
    }

    with torch.inference_mode():  # Turn off training calculations
        token_embeddings = encoder_model(**encoder_inputs).last_hidden_state  # Get token embeddings

    token_embeddings = token_embeddings.detach().float().cpu()  # Move results to the CPU
    encoder_ids = encoder_inputs["input_ids"][0].detach().cpu().tolist()  # Get token IDs
    encoder_tokens = encoder_tokenizer.convert_ids_to_tokens(encoder_ids)  # Get readable token names

    embedding_size = token_embeddings.shape[-1]  # Count dimensions per token
    embedding_table = pd.DataFrame(  # Create a table of embeddings
        token_embeddings[0].numpy(),  # Include every token and every dimension
        index=encoder_tokens,  # Label rows with token names
        columns=[f"dimension_{i}" for i in range(embedding_size)],  # Label all dimensions
    )

In [19]:
if embedding_table is not None:
    print(f"Embedding tensor shape: {tuple(token_embeddings.shape)}")
    with pd.option_context("display.max_columns", None):
        display(embedding_table)
else:
    print("Contextual embedding example skipped because the encoder is unavailable.")

Embedding tensor shape: (1, 4, 384)


,dimension_0,dimension_1,dimension_2,dimension_3,dimension_4,dimension_5,dimension_6,dimension_7,dimension_8,dimension_9,dimension_10,dimension_11,dimension_12,dimension_13,dimension_14,dimension_15,dimension_16,dimension_17,dimension_18,dimension_19,dimension_20,dimension_21,dimension_22,dimension_23,dimension_24,dimension_25,dimension_26,dimension_27,dimension_28,dimension_29,dimension_30,dimension_31,dimension_32,dimension_33,dimension_34,dimension_35,dimension_36,dimension_37,dimension_38,dimension_39,dimension_40,dimension_41,dimension_42,dimension_43,dimension_44,dimension_45,dimension_46,dimension_47,dimension_48,dimension_49,dimension_50,dimension_51,dimension_52,dimension_53,dimension_54,dimension_55,dimension_56,dimension_57,dimension_58,dimension_59,dimension_60,dimension_61,dimension_62,dimension_63,dimension_64,dimension_65,dimension_66,dimension_67,dimension_68,dimension_69,dimension_70,dimension_71,dimension_72,dimension_73,dimension_74,dimension_75,dimension_76,dimension_77,dimension_78,dimension_79,dimension_80,dimension_81,dimension_82,dimension_83,dimension_84,dimension_85,dimension_86,dimension_87,dimension_88,dimension_89,dimension_90,dimension_91,dimension_92,dimension_93,dimension_94,dimension_95,dimension_96,dimension_97,dimension_98,dimension_99,dimension_100,dimension_101,dimension_102,dimension_103,dimension_104,dimension_105,dimension_106,dimension_107,dimension_108,dimension_109,dimension_110,dimension_111,dimension_112,dimension_113,dimension_114,dimension_115,dimension_116,dimension_117,dimension_118,dimension_119,dimension_120,dimension_121,dimension_122,dimension_123,dimension_124,dimension_125,dimension_126,dimension_127,dimension_128,dimension_129,dimension_130,dimension_131,dimension_132,dimension_133,dimension_134,dimension_135,dimension_136,dimension_137,dimension_138,dimension_139,dimension_140,dimension_141,dimension_142,dimension_143,dimension_144,dimension_145,dimension_146,dimension_147,dimension_148,dimension_149,dimension_150,dimension_151,dimension_152,dimension_153,dimension_154,dimension_155,dimension_156,dimension_157,dimension_158,dimension_159,dimension_160,dimension_161,dimension_162,dimension_163,dimension_164,dimension_165,dimension_166,dimension_167,dimension_168,dimension_169,dimension_170,dimension_171,dimension_172,dimension_173,dimension_174,dimension_175,dimension_176,dimension_177,dimension_178,dimension_179,dimension_180,dimension_181,dimension_182,dimension_183,dimension_184,dimension_185,dimension_186,dimension_187,dimension_188,dimension_189,dimension_190,dimension_191,dimension_192,dimension_193,dimension_194,dimension_195,dimension_196,dimension_197,dimension_198,dimension_199,dimension_200,dimension_201,dimension_202,dimension_203,dimension_204,dimension_205,dimension_206,dimension_207,dimension_208,dimension_209,dimension_210,dimension_211,dimension_212,dimension_213,dimension_214,dimension_215,dimension_216,dimension_217,dimension_218,dimension_219,dimension_220,dimension_221,dimension_222,dimension_223,dimension_224,dimension_225,dimension_226,dimension_227,dimension_228,dimension_229,dimension_230,dimension_231,dimension_232,dimension_233,dimension_234,dimension_235,dimension_236,dimension_237,dimension_238,dimension_239,dimension_240,dimension_241,dimension_242,dimension_243,dimension_244,dimension_245,dimension_246,dimension_247,dimension_248,dimension_249,dimension_250,dimension_251,dimension_252,dimension_253,dimension_254,dimension_255,dimension_256,dimension_257,dimension_258,dimension_259,dimension_260,dimension_261,dimension_262,dimension_263,dimension_264,dimension_265,dimension_266,dimension_267,dimension_268,dimension_269,dimension_270,dimension_271,dimension_272,dimension_273,dimension_274,dimension_275,dimension_276,dimension_277,dimension_278,dimension_279,dimension_280,dimension_281,dimension_282,dimension_283,dimension_284,dimension_285,dimension_286,dimension_287,dimension_288,dimension_289,dimension_290,dimension_291,dimension_292,dimensi

The shape is **(batch size, token positions, hidden size)**. For `Hello world`, DeBERTa produces `(1, 4, 384)`: one text, four positions including special tokens, and 384 values per position.

## Part IV. Song embeddings from playlists

Embeddings can represent items beyond text. Treat a playlist like a sentence and each song ID like a word: Word2Vec can learn song vectors from co-occurrence. The cells below inspect the data used for that approach. A few titles/artists are missing; song IDs still identify the tracks.

Full training example: [Word Embeddings Beyond LLMs, Chapter 2 companion notebook](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb).

In [ ]:
# Skip the first two metadata lines; keep playlists with at least two songs.
with open("song-playlists.txt", encoding="utf-8") as file:
    playlist_lines = file.read().splitlines()[2:]
playlists = [line.split() for line in playlist_lines if len(line.split()) > 1]

songs_df = pd.read_csv(
    "song_hash.txt",
    sep="\t",
    names=["id", "title", "artist"],
    dtype=str,
    index_col="id",
)

# The metadata file has spaces around its song IDs; match the playlist IDs.
songs_df.index = songs_df.index.str.strip()

print(f"Usable playlists: {len(playlists):,}")
print(f"Songs with metadata: {len(songs_df):,}")


In [ ]:
playlist_lengths = pd.Series(
    [len(playlist) for playlist in playlists], name="songs_per_playlist"
)
display(playlist_lengths.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(1))

for playlist_number in range(2):
    example_ids = playlists[playlist_number][:10]
    print(
        f"Playlist {playlist_number + 1}: {len(playlists[playlist_number])} songs; "
        "first 10 shown below"
    )
    display(songs_df.reindex(example_ids))


## Representation reference

| Representation | Unit represented | Learned from |
|---|---|---|
| Token ID | Vocabulary symbol (a lookup key, not an embedding) | Tokenizer vocabulary |
| Contextual token embedding | Token at one position | Transformer training |
| Sentence embedding | Complete text | Similarity-oriented model training |
| Song embedding | Song ID | Playlist co-occurrence |

Adapted from Serena Kim's DSA 495 Module 2 notebook and Alammar and Grootendorst, *Hands-On Large Language Models*, Chapter 2. Model references: [DeBERTa](https://huggingface.co/microsoft/deberta-v3-xsmall), [MPNet](https://huggingface.co/sentence-transformers/all-mpnet-base-v2), and [MiniLM](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2).